In [1]:
import os
import numpy as np
from PIL import Image
import monai
from sklearn.utils import resample

import torch
from torch.utils.data import Dataset
from torch.nn import functional as F

/home/zhenyi/anaconda3/envs/RETFound_Segmenter/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class DatasetFromRoot(Dataset):
    def __init__(self, gt_root_path, pred_root_path):

        self.gt_all = []
        self.pred_all = []
        self.list = []
        
        gt_list = os.listdir(gt_root_path)
        gt_list.sort()       
        for i in gt_list:
            gt_path = gt_root_path + os.sep + i
            gt = np.asarray(Image.open(gt_path)).astype(int)
            self.gt_all.append(gt)
        
        pred_list = os.listdir(pred_root_path)
        pred_list.sort()       
        for j in pred_list:
            pred_path = pred_root_path + os.sep + j
            if pred_path[-4:] == '.png':
                pred = np.asarray(Image.open(pred_path)).astype(int)
                self.pred_all.append(pred)
                self.list.append(j)

        
    def __len__(self):
        return len(self.gt_all)

    def __getitem__(self, idx):
        gt = self.gt_all[idx] 
        pred = self.pred_all[idx]
        filename = self.list[idx]
        gt = torch.tensor(gt)
        pred = torch.tensor(pred)
        gt = gt.unsqueeze(0)
        gt = gt.long()
        pred = pred.unsqueeze(0)
        pred = pred.long()
        return gt, pred, filename

In [3]:
test_Drishti_GS = DatasetFromRoot(gt_root_path = '/home/zhenyi/Project/nnUNet/nnUNet_Drishti_GS/nnUNet_raw/Dataset001_Drishti_GS/labelsTs', 
                                  pred_root_path = '/home/zhenyi/Project/nnUNet/nnUNet_IDRID/output_Drishti_GS')

test_Vampire = DatasetFromRoot(gt_root_path = '/home/zhenyi/Project/nnUNet/nnUNet_Vampire/nnUNet_raw/Dataset001_Vampire/labelsTs', 
                                  pred_root_path = '/home/zhenyi/Project/nnUNet/nnUNet_IDRID/output_Vampire')

test_IDRID = DatasetFromRoot(gt_root_path = '/home/zhenyi/Project/nnUNet/nnUNet_IDRID/nnUNet_raw/Dataset001_IDRID/labelsTs', 
                                  pred_root_path = '/home/zhenyi/Project/nnUNet/nnUNet_IDRID/output_IDRID')

test_RIM_ONE_r3 = DatasetFromRoot(gt_root_path = '/home/zhenyi/Project/nnUNet/nnUNet_RIM_ONE_r3/nnUNet_raw/Dataset001_RIM_ONE_r3/labelsTs', 
                                  pred_root_path = '/home/zhenyi/Project/nnUNet/nnUNet_IDRID/output_RIM_ONE_r3')

test_REFUGE = DatasetFromRoot(gt_root_path = '/home/zhenyi/Project/nnUNet/nnUNet_REFUGE/nnUNet_raw/Dataset001_REFUGE/labelsTs', 
                                  pred_root_path = '/home/zhenyi/Project/nnUNet/nnUNet_IDRID/output_REFUGE')

In [4]:
od_dice_metrics = monai.metrics.DiceMetric(include_background=True, reduction='none')

for i in range(0, len(test_IDRID)):

    y_true, y_pred = test_IDRID[i][0].unsqueeze(dim=0), test_IDRID[i][1].unsqueeze(dim=0)
    y_true = F.one_hot(y_true.squeeze(dim=1), num_classes=2).permute(0,3,1,2)
    y_pred = F.one_hot(y_pred.squeeze(dim=1), num_classes=2).permute(0,3,1,2)
    
    od_dice_metrics(y_pred[:,1,:,:].unsqueeze(dim=1), y_true[:,1,:,:].unsqueeze(dim=1))

dice_od_all = np.asarray(od_dice_metrics.aggregate().squeeze(dim=1))

dice_od_mean = np.mean(dice_od_all)

print('IDRID od Dice is:', dice_od_mean)

min_index = np.argmin(dice_od_all)
max_index = np.argmax(dice_od_all)
print('the worst is:', test_IDRID[min_index][2])
print('the best is:', test_IDRID[max_index][2])

seed = np.random.RandomState(112316)
n_iterations = 10000
alpha = 0.95
bootstrap_od_means = []
for i in range(n_iterations):
    bootstrap_od_sample = resample(dice_od_all, n_samples=len(dice_od_all), random_state=seed)
    bootstrap_od_means.append(np.mean(bootstrap_od_sample))
    
lower = ((1.0-alpha)/2.0)*100
upper = (alpha+((1.0-alpha)/2.0))*100

od_ci = np.percentile(bootstrap_od_means, [lower, upper])

print('od CI is:', od_ci)

IDRID od Dice is: 0.9699023
the worst is: IDRiD_60_OD.png
the best is: IDRiD_76_OD.png
od CI is: [0.96298786 0.97580379]


In [5]:
od_dice_metrics = monai.metrics.DiceMetric(include_background=True, reduction='none')

for i in range(0, len(test_Vampire)):

    y_true, y_pred = test_Vampire[i][0].unsqueeze(dim=0), test_Vampire[i][1].unsqueeze(dim=0)
    y_true = F.one_hot(y_true.squeeze(dim=1), num_classes=2).permute(0,3,1,2)
    y_pred = F.one_hot(y_pred.squeeze(dim=1), num_classes=2).permute(0,3,1,2)
    
    od_dice_metrics(y_pred[:,1,:,:].unsqueeze(dim=1), y_true[:,1,:,:].unsqueeze(dim=1))

dice_od_all = np.asarray(od_dice_metrics.aggregate().squeeze(dim=1))

dice_od_mean = np.mean(dice_od_all)

print('Vampire od Dice is:', dice_od_mean)

min_index = np.argmin(dice_od_all)
max_index = np.argmax(dice_od_all)
print('the worst is:', test_Vampire[min_index][2])
print('the best is:', test_Vampire[max_index][2])

seed = np.random.RandomState(112316)
n_iterations = 10000
alpha = 0.95
bootstrap_od_means = []
for i in range(n_iterations):
    bootstrap_od_sample = resample(dice_od_all, n_samples=len(dice_od_all), random_state=seed)
    bootstrap_od_means.append(np.mean(bootstrap_od_sample))
    
lower = ((1.0-alpha)/2.0)*100
upper = (alpha+((1.0-alpha)/2.0))*100

od_ci = np.percentile(bootstrap_od_means, [lower, upper])

print('od CI is:', od_ci)

Vampire od Dice is: 0.9311043
the worst is: 15.png
the best is: 30.png
od CI is: [0.91626944 0.94558072]


In [6]:
od_dice_metrics = monai.metrics.DiceMetric(include_background=True, reduction='none')

for i in range(0, len(test_Drishti_GS)):

    y_true, y_pred = test_Drishti_GS[i][0].unsqueeze(dim=0), test_Drishti_GS[i][1].unsqueeze(dim=0)
    y_true = F.one_hot(y_true.squeeze(dim=1), num_classes=3).permute(0,3,1,2)
    y_pred = F.one_hot(y_pred.squeeze(dim=1), num_classes=2).permute(0,3,1,2)
    
    od_dice_metrics(y_pred[:,1,:,:].unsqueeze(dim=1), (y_true[:,1,:,:]+y_true[:,2,:,:]).unsqueeze(dim=1))

dice_od_all = np.asarray(od_dice_metrics.aggregate().squeeze(dim=1))

dice_od_mean = np.mean(dice_od_all)

print('Drishti_GS od Dice is:', dice_od_mean)

min_index = np.argmin(dice_od_all)
max_index = np.argmax(dice_od_all)
print('the worst is:', test_Drishti_GS[min_index][2])
print('the best is:', test_Drishti_GS[max_index][2])

seed = np.random.RandomState(112316)
n_iterations = 10000
alpha = 0.95
bootstrap_od_means = []
for i in range(n_iterations):
    bootstrap_od_sample = resample(dice_od_all, n_samples=len(dice_od_all), random_state=seed)
    bootstrap_od_means.append(np.mean(bootstrap_od_sample))
    
lower = ((1.0-alpha)/2.0)*100
upper = (alpha+((1.0-alpha)/2.0))*100

od_ci = np.percentile(bootstrap_od_means, [lower, upper])

print('od CI is:', od_ci)

Drishti_GS od Dice is: 0.9297188
the worst is: drishtiGS_023.png
the best is: drishtiGS_007.png
od CI is: [0.89007994 0.95568883]


In [7]:
od_dice_metrics = monai.metrics.DiceMetric(include_background=True, reduction='none')

for i in range(0, len(test_RIM_ONE_r3)):

    y_true, y_pred = test_RIM_ONE_r3[i][0].unsqueeze(dim=0), test_RIM_ONE_r3[i][1].unsqueeze(dim=0)
    y_true = F.one_hot(y_true.squeeze(dim=1), num_classes=3).permute(0,3,1,2)
    y_pred = F.one_hot(y_pred.squeeze(dim=1), num_classes=2).permute(0,3,1,2)
    
    od_dice_metrics(y_pred[:,1,:,:].unsqueeze(dim=1), (y_true[:,1,:,:]+y_true[:,2,:,:]).unsqueeze(dim=1))

dice_od_all = np.asarray(od_dice_metrics.aggregate().squeeze(dim=1))

dice_od_mean = np.mean(dice_od_all)

print('RIM_ONE_r3 od Dice is:', dice_od_mean)

min_index = np.argmin(dice_od_all)
max_index = np.argmax(dice_od_all)
print('the worst is:', test_RIM_ONE_r3[min_index][2])
print('the best is:', test_RIM_ONE_r3[max_index][2])

seed = np.random.RandomState(112316)
n_iterations = 10000
alpha = 0.95
bootstrap_od_means = []
for i in range(n_iterations):
    bootstrap_od_sample = resample(dice_od_all, n_samples=len(dice_od_all), random_state=seed)
    bootstrap_od_means.append(np.mean(bootstrap_od_sample))
    
lower = ((1.0-alpha)/2.0)*100
upper = (alpha+((1.0-alpha)/2.0))*100

od_ci = np.percentile(bootstrap_od_means, [lower, upper])

print('od CI is:', od_ci)

RIM_ONE_r3 od Dice is: 0.79065406
the worst is: G-31-R.png
the best is: G-30-L.png
od CI is: [0.72952505 0.84141608]


In [8]:
od_dice_metrics = monai.metrics.DiceMetric(include_background=True, reduction='none')

for i in range(0, len(test_REFUGE)):

    y_true, y_pred = test_REFUGE[i][0].unsqueeze(dim=0), test_REFUGE[i][1].unsqueeze(dim=0)
    y_true = F.one_hot(y_true.squeeze(dim=1), num_classes=3).permute(0,3,1,2)
    y_pred = F.one_hot(y_pred.squeeze(dim=1), num_classes=2).permute(0,3,1,2)
    
    od_dice_metrics(y_pred[:,1,:,:].unsqueeze(dim=1), (y_true[:,1,:,:]+y_true[:,2,:,:]).unsqueeze(dim=1))

dice_od_all = np.asarray(od_dice_metrics.aggregate().squeeze(dim=1))

dice_od_mean = np.mean(dice_od_all)

print('REFUGE od Dice is:', dice_od_mean)

min_index = np.argmin(dice_od_all)
max_index = np.argmax(dice_od_all)
print('the worst is:', test_REFUGE[min_index][2])
print('the best is:', test_REFUGE[max_index][2])

seed = np.random.RandomState(112316)
n_iterations = 10000
alpha = 0.95
bootstrap_od_means = []
for i in range(n_iterations):
    bootstrap_od_sample = resample(dice_od_all, n_samples=len(dice_od_all), random_state=seed)
    bootstrap_od_means.append(np.mean(bootstrap_od_sample))
    
lower = ((1.0-alpha)/2.0)*100
upper = (alpha+((1.0-alpha)/2.0))*100

od_ci = np.percentile(bootstrap_od_means, [lower, upper])

print('od CI is:', od_ci)

REFUGE od Dice is: 0.6321298
the worst is: T0002.png
the best is: T0115.png
od CI is: [0.593339   0.66988215]
